# Building a first Classification NN

Use the Ames Mutagenicity dataset (from assignment 1A) and build a binary classifier NN. Play with the model parameters. 

For comparison of the NN model performance, consider the performance of other (baseline) classifier models (assignment 1A):
- KNN: Test-Accuracy 0.79, Test-ROC-AUC 0.86
- Decision Tree: Test-Accuracy 0.78, Test-ROC-AUC 0.77
- Random Forest: Test-Accuracy 0.83, Test-ROC-AUC 0.90
- Gradient Boosting: Test-Accuracy 0.77, Test-ROC-AUC 0.85


#### Tasks:
1) Load the dataset `ames_data.csv`. The dataset does not contain any duplicates or NaNs
2) Feature engineering: Calculate various fingerprints from the SMILES strings via mol objects using RDKit(snippet provided for Morgan FPs and MACCS keys)
3) Create feature matrix and target vector. Choose first the MorganFP (Later repeat the process for other fingerprint types). Convert the training and test sets into pytorch tensors.
4) Build your NN (see below for more info)
5) Train your model on the Morgan Fingerprints (and repeat later for other FP types)
6) Evaluate your model's performance and compare to other classifier models
7) Save the model / current state.
8) Respond to the discussion points


#### Note:
The aim of this exercise is to gain a bit of practice in building a simple NN and to see how different parameters and feature engineering influence the model. Maximum accuracy is not the target. 

There is no need to venture too far into the details or more advanced approaches just yet (e.g. batched training would be complete overkill for this assignment - we will discuss that in the next sessions)

0) Import dependencies and datasets

In [31]:
# complete imports if needed for your solution
import pandas as pd
import numpy as np

from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from rdkit.Chem import MACCSkeys

from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score, roc_auc_score

1) Load and investigate the data

In [2]:
df = pd.read_csv("ames_data.csv")
df.head()

,drug_id,smiles,mutagenicity
0,Drug 0,O=[N+]([O-])c1ccc2ccc3ccc([N+](=O)[O-])c4c5ccc...,1
1,Drug 1,O=[N+]([O-])c1c2c(c3ccc4cccc5ccc1c3c45)CCCC2,1
2,Drug 2,O=c1c2ccccc2c(=O)c2c1ccc1c2[nH]c2c3c(=O)c4cccc...,0
3,Drug 3,[N-]=[N+]=CC(=O)NCC(=O)NN,1
4,Drug 4,[N-]=[N+]=C1C=NC(=O)NC1=O,1


2) Generate different fingerprints (try at least one additional FP type as provided in RDKit and use two different fpSizes on one of them) - all of them will be saved in new columns in the Dataframe.

In [4]:
def smiles_to_mol(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return mol

def morganfp(mol):
    fp = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048).GetFingerprint(mol)
    return np.array(fp)

def morganfp_1024(mol):
    fp = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024).GetFingerprint(mol)
    return np.array(fp)

def maccskeys(mol):
    fp = MACCSkeys.GenMACCSKeys(mol)
    return np.array(fp)

def topological_torsions(mol):
    fp = rdFingerprintGenerator.GetTopologicalTorsionGenerator(fpSize=2048).GetFingerprint(mol)
    return np.array(fp)

def topological_torsions_1024(mol):
    fp = rdFingerprintGenerator.GetTopologicalTorsionGenerator(fpSize=1024).GetFingerprint(mol)
    return np.array(fp)

fpgens = {
    "MorganFP": morganfp,
    "MorganFP_1024": morganfp_1024,
    "MACCSkeys": maccskeys,
    "TopologicalTorsions": topological_torsions,
    "TopologicalTorsions_1024": topological_torsions_1024
}

df["mol"] = df["smiles"].apply(smiles_to_mol)

for name, fpgen in fpgens.items():
    df[name] = df["mol"].apply(fpgen)
df.head()

,drug_id,smiles,mutagenicity,mol,MorganFP,MorganFP_1024,MACCSkeys,TopologicalTorsions,TopologicalTorsions_1024
0,Drug 0,O=[N+]([O-])c1ccc2ccc3ccc([N+](=O)[O-])c4c5ccc...,1,<rdkit.Chem.rdchem.Mol object at 0x11dc4d230>,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."
1,Drug 1,O=[N+]([O-])c1c2c(c3ccc4cccc5ccc1c3c45)CCCC2,1,<rdkit.Chem.rdchem.Mol object at 0x11dc4e500>,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,Drug 2,O=c1c2ccccc2c(=O)c2c1ccc1c2[nH]c2c3c(=O)c4cccc...,0,<rdkit.Chem.rdchem.Mol object at 0x11dc4dc40>,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, ..."
3,Drug 3,[N-]=[N+]=CC(=O)NCC(=O)NN,1,<rdkit.Chem.rdchem.Mol object at 0x11dc4e340>,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,Drug 4,[N-]=[N+]=C1C=NC(=O)NC1=O,1,<rdkit.Chem.rdchem.Mol object at 0x11dc4e5e0>,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


3. Create feature matrix and target vector. The snippet below converts the data into numpy arrays. Start with the Morgan Fingerprints (and later return here to apply your modell to different fingerprint types - not all of the fingerprints may have the same length, so you may have to adapt the width of your layers).

Do a train startified test split and convert into pytorch tensors.

In [24]:
X = np.stack(df["MorganFP"].values) # joins multiple arrays along a new axis - builds a proper array from the line-by-line arrays in the df.
y = df["mutagenicity"].to_numpy()

X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32)

4) Build the NN - adhere to some robust standard values. Start simple and train the model on Morgan FP first.

Optimise the model parameters based on observed over-/underfitting. Experiment with different width and depth, as well as other model parameters. Explore some options to prevent overfitting, e.g. Early stopping (e.g. manually by limiting the epochs) or dropouts. 

Note: Since the input layer needs a lot of neurons (e.g. 2048 bit in the MFPs), consider shrinking the widht from layer to layer. 

Hint: If you use `BCELoss()` as loss function, combine it with a `sigmoid` activation in the last layer. If you use `BCEWithLogitsLoss()`, do not specify any activation in the forward pass (`x = self.outputlayer(x)`).

In [ ]:
class BinClassifierNN(nn.Module):
    def __init__(self):
        super(BinClassifierNN, self).__init__()
        # define your model width and depth below
        self.fc1 = nn.Linear(2048, 4096)  # 2048 input nodes to 4096 hidden nodes (fc: fully connected)
        self.fch = nn.Linear(4096, 4096)  # 4096 hidden nodes to 4096 hidden nodes
        self.fc2 = nn.Linear(4096, 1)  # 4096 hidden nodes to 1 output node (binary classification)

    def forward(self, x):
        # Specify the forward pass, i.e. activation functions.
        x = torch.relu(self.fc1(x))  # activation function
        x = torch.relu(self.fch(x))
        x = torch.sigmoid(self.fc2(x))

        return x


In [97]:
# Parameters (change and add as needed)
learning_rate = 0.8
num_epochs = 200

In [98]:
model = BinClassifierNN()

# choose a loss function for the classification
criterion = nn.BCELoss() 

# choose an optimizer
optimizer = optim.SGD(model.parameters(), lr=learning_rate)

5. Train the NN. Note that you may have to squeeze the output (`outputs=models(X_train).squeeze`). This will reduce the actual output of the shape ``[N, 1]`` to ``[N]``, which is comparable to y (The final layer naturally produces a column tensor, which is not directly comparable to the 1D target tensor).

In [99]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


for epoch in range(num_epochs):
    
    model.train()
    outputs = model(X_train).squeeze()
    loss = criterion(outputs, y_train)

    optimizer.zero_grad() # clear existing gradients
    loss.backward() # backpropagation of loss function to optimise gradient
    optimizer.step() # update parameters using Adam

    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch + 1}/100], Loss: {loss.item():.4f}')

Epoch [10/100], Loss: 0.6630
Epoch [20/100], Loss: 0.5945
Epoch [30/100], Loss: 0.5239
Epoch [40/100], Loss: 0.5668
Epoch [50/100], Loss: 0.4994
Epoch [60/100], Loss: 0.4392
Epoch [70/100], Loss: 0.5268
Epoch [80/100], Loss: 0.4280
Epoch [90/100], Loss: 0.5692
Epoch [100/100], Loss: 0.3876
Epoch [110/100], Loss: 0.4099
Epoch [120/100], Loss: 0.3973
Epoch [130/100], Loss: 0.3716
Epoch [140/100], Loss: 0.3391
Epoch [150/100], Loss: 0.3127
Epoch [160/100], Loss: 0.3245
Epoch [170/100], Loss: 0.4322
Epoch [180/100], Loss: 0.2316
Epoch [190/100], Loss: 0.4012
Epoch [200/100], Loss: 0.2140


6) Evaluate the model. As a first metric, you can use the same loss function to evaluate the model on the test set. For better comparison with methods tested in the assignment 1A (results vide supra), use metrics from scikit-learn (e.g. the accuracy or ROC-AUC score).

Hint: for your prediction you may have to use `squeeze` again to match the target vector in the test set (e.g. ``y_pred = model(X_test).squeeze()``)

In [100]:
y_train_pred = ((model(X_train).squeeze() > 0.5).float()).detach().numpy()
y_test_pred = ((model(X_test).squeeze() > 0.5).float()).detach().numpy()

In [101]:
print(y_train)
print(y_train_pred)

tensor([1., 1., 0.,  ..., 1., 0., 1.])
[1. 1. 0. ... 0. 0. 1.]


In [102]:
print("Accuracy and ROC AUC")
print("Accuracy on training set:", accuracy_score(y_train, y_train_pred))
print("Accuracy on test set:", accuracy_score(y_test, y_test_pred))
print("ROC AUC on training set:", roc_auc_score(y_train, y_train_pred))
print("ROC AUC on test set:", roc_auc_score(y_test, y_test_pred))

Accuracy and ROC AUC
Accuracy on training set: 0.9237375472346273
Accuracy on test set: 0.8337912087912088
ROC AUC on training set: 0.9233525344086698
ROC AUC on test set: 0.8315736148882846


Results with parameters (0.01 and 100)
- Accuracy and ROC AUC
- Accuracy on training set: 0.5455170044658193
- Accuracy on test set: 0.5480769230769231
- ROC AUC on training set: 0.5
- ROC AUC on test set: 0.5

Results with parameters (0.1 and 100)
- Accuracy and ROC AUC
- Accuracy on training set: 0.6442803160425971
- Accuracy on test set: 0.6401098901098901
- ROC AUC on training set: 0.6188084385727423
- ROC AUC on test set: 0.6122220444728843

Results with parameters (0.4 and 100)
- Accuracy and ROC AUC
- Accuracy on training set: 0.8193060803847475
- Accuracy on test set: 0.7850274725274725
- ROC AUC on training set: 0.8183006619944366
- ROC AUC on test set: 0.7833546632538794

Results with parameters (0.8 and 100)
- Accuracy and ROC AUC
- Accuracy on training set: 0.8400893163861216
- Accuracy on test set: 0.7946428571428571
- ROC AUC on training set: 0.8400932161277609
- ROC AUC on test set: 0.7943928971364582

Results with parameters (0.6 and 100)
- Accuracy and ROC AUC
- Accuracy on training set: 0.7986946066643765
- Accuracy on test set: 0.7651098901098901
- ROC AUC on training set: 0.8077654165500514
- ROC AUC on test set: 0.7755825734549139

Results with parameters (1 and 100)
- Accuracy and ROC AUC
- Accuracy on training set: 0.7638268636207489
- Accuracy on test set: 0.7115384615384616
- ROC AUC on training set: 0.7814515184747282
- ROC AUC on test set: 0.7312430011198208

Results with parameters (0.7 and 100)
- Accuracy and ROC AUC
- Accuracy on training set: 0.8337341119890073
- Accuracy on test set: 0.7939560439560439
- ROC AUC on training set: 0.8330699968204466
- ROC AUC on test set: 0.7924332106862904

Results with parameters (0.8 and 50)
- Accuracy and ROC AUC
- Accuracy on training set: 0.780831329440055
- Accuracy on test set: 0.7548076923076923
- ROC AUC on training set: 0.7817432948550258
- ROC AUC on test set: 0.7564523009651789

Results with parameters (0.8 and 200)
- Accuracy and ROC AUC
- Accuracy on training set: 0.9237375472346273
- Accuracy on test set: 0.8337912087912088
- ROC AUC on training set: 0.9233525344086698
- ROC AUC on test set: 0.8315736148882846

7) Research how you can save the model / current state for later reuse. What are different options here? How can it be loaded again?

In [103]:
torch.save(model.state_dict(), "model.pth")


In [104]:
model = BinClassifierNN()
model.load_state_dict(torch.load("model.pth"))
model.eval()

BinClassifierNN(
  (fc1): Linear(in_features=2048, out_features=4096, bias=True)
  (fch): Linear(in_features=4096, out_features=4096, bias=True)
  (fc2): Linear(in_features=4096, out_features=1, bias=True)
)

In [105]:
# redo with different fingerprints

X = np.stack(df["MorganFP_1024"].values) # joins multiple arrays along a new axis - builds a proper array from the line-by-line arrays in the df.
y = df["mutagenicity"].to_numpy()

X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32)

class BinClassifierNN(nn.Module):
    def __init__(self):
        super(BinClassifierNN, self).__init__()
        # define your model width and depth below
        self.fc1 = nn.Linear(1024, 4096)  # 1024 input nodes to 4096 hidden nodes (fc: fully connected)
        self.fch = nn.Linear(4096, 4096)  # 4096 hidden nodes to 4096 hidden nodes
        self.fc2 = nn.Linear(4096, 1)  # 4096 hidden nodes to 1 output node (binary classification)

    def forward(self, x):
        # Specify the forward pass, i.e. activation functions.
        x = torch.relu(self.fc1(x))  # activation function
        x = torch.relu(self.fch(x))
        x = torch.sigmoid(self.fc2(x))

        return x


# Parameters (change and add as needed)
learning_rate = 0.8
num_epochs = 200

model = BinClassifierNN()

# choose a loss function for the classification
criterion = nn.BCELoss() 

# choose an optimizer
optimizer = optim.SGD(model.parameters(), lr=learning_rate)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


for epoch in range(num_epochs):
    
    model.train()
    outputs = model(X_train).squeeze()
    loss = criterion(outputs, y_train)

    optimizer.zero_grad() # clear existing gradients
    loss.backward() # backpropagation of loss function to optimise gradient
    optimizer.step() # update parameters using Adam

    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch + 1}/100], Loss: {loss.item():.4f}')

y_train_pred = ((model(X_train).squeeze() > 0.5).float()).detach().numpy()
y_test_pred = ((model(X_test).squeeze() > 0.5).float()).detach().numpy()

print("Accuracy and ROC AUC")
print("Accuracy on training set:", accuracy_score(y_train, y_train_pred))
print("Accuracy on test set:", accuracy_score(y_test, y_test_pred))
print("ROC AUC on training set:", roc_auc_score(y_train, y_train_pred))
print("ROC AUC on test set:", roc_auc_score(y_test, y_test_pred))

Epoch [10/100], Loss: 0.6486
Epoch [20/100], Loss: 0.5713
Epoch [30/100], Loss: 0.6659
Epoch [40/100], Loss: 0.5424
Epoch [50/100], Loss: 0.4820
Epoch [60/100], Loss: 0.5075
Epoch [70/100], Loss: 0.4277
Epoch [80/100], Loss: 0.5808
Epoch [90/100], Loss: 0.4524
Epoch [100/100], Loss: 0.3917
Epoch [110/100], Loss: 1.0065
Epoch [120/100], Loss: 0.3973
Epoch [130/100], Loss: 0.3323
Epoch [140/100], Loss: 1.1803
Epoch [150/100], Loss: 0.3228
Epoch [160/100], Loss: 0.2639
Epoch [170/100], Loss: 0.5399
Epoch [180/100], Loss: 0.2792
Epoch [190/100], Loss: 0.2146
Epoch [200/100], Loss: 0.3374
Accuracy and ROC AUC
Accuracy on training set: 0.9017519752662315
Accuracy on test set: 0.8221153846153846
ROC AUC on training set: 0.9015616462090015
ROC AUC on test set: 0.8206553618087773


In [108]:
# redo with different fingerprints

X = np.stack(df["MACCSkeys"].values) # joins multiple arrays along a new axis - builds a proper array from the line-by-line arrays in the df.
y = df["mutagenicity"].to_numpy()

X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32)

class BinClassifierNN(nn.Module):
    def __init__(self):
        super(BinClassifierNN, self).__init__()
        # define your model width and depth below
        self.fc1 = nn.Linear(167, 4096)  # 167 input nodes to 4096 hidden nodes (fc: fully connected)
        self.fch = nn.Linear(4096, 4096)  # 4096 hidden nodes to 4096 hidden nodes
        self.fc2 = nn.Linear(4096, 1)  # 4096 hidden nodes to 1 output node (binary classification)

    def forward(self, x):
        # Specify the forward pass, i.e. activation functions.
        x = torch.relu(self.fc1(x))  # activation function
        x = torch.relu(self.fch(x))
        x = torch.sigmoid(self.fc2(x))

        return x


# Parameters (change and add as needed)
learning_rate = 0.8
num_epochs = 200

model = BinClassifierNN()

# choose a loss function for the classification
criterion = nn.BCELoss() 

# choose an optimizer
optimizer = optim.SGD(model.parameters(), lr=learning_rate)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


for epoch in range(num_epochs):
    
    model.train()
    outputs = model(X_train).squeeze()
    loss = criterion(outputs, y_train)

    optimizer.zero_grad() # clear existing gradients
    loss.backward() # backpropagation of loss function to optimise gradient
    optimizer.step() # update parameters using Adam

    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch + 1}/100], Loss: {loss.item():.4f}')

y_train_pred = ((model(X_train).squeeze() > 0.5).float()).detach().numpy()
y_test_pred = ((model(X_test).squeeze() > 0.5).float()).detach().numpy()

print("Accuracy and ROC AUC")
print("Accuracy on training set:", accuracy_score(y_train, y_train_pred))
print("Accuracy on test set:", accuracy_score(y_test, y_test_pred))
print("ROC AUC on training set:", roc_auc_score(y_train, y_train_pred))
print("ROC AUC on test set:", roc_auc_score(y_test, y_test_pred))

Epoch [10/100], Loss: 0.6154
Epoch [20/100], Loss: 0.7052
Epoch [30/100], Loss: 0.5856
Epoch [40/100], Loss: 0.6234
Epoch [50/100], Loss: 0.5443
Epoch [60/100], Loss: 0.6584
Epoch [70/100], Loss: 0.5314
Epoch [80/100], Loss: 0.5234
Epoch [90/100], Loss: 0.4948
Epoch [100/100], Loss: 0.4814
Epoch [110/100], Loss: 0.4548
Epoch [120/100], Loss: 0.4760
Epoch [130/100], Loss: 0.4398
Epoch [140/100], Loss: 0.6286
Epoch [150/100], Loss: 0.4367
Epoch [160/100], Loss: 0.4114
Epoch [170/100], Loss: 0.4101
Epoch [180/100], Loss: 0.3920
Epoch [190/100], Loss: 0.3971
Epoch [200/100], Loss: 0.3792
Accuracy and ROC AUC
Accuracy on training set: 0.8086568189625558
Accuracy on test set: 0.7802197802197802
ROC AUC on training set: 0.7948543117218898
ROC AUC on test set: 0.7660374340105582


In [109]:
# redo with different fingerprints

X = np.stack(df["TopologicalTorsions"].values) # joins multiple arrays along a new axis - builds a proper array from the line-by-line arrays in the df.
y = df["mutagenicity"].to_numpy()

X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32)

class BinClassifierNN(nn.Module):
    def __init__(self):
        super(BinClassifierNN, self).__init__()
        # define your model width and depth below
        self.fc1 = nn.Linear(2048, 4096)  # 2048 input nodes to 4096 hidden nodes (fc: fully connected)
        self.fch = nn.Linear(4096, 4096)  # 4096 hidden nodes to 4096 hidden nodes
        self.fc2 = nn.Linear(4096, 1)  # 4096 hidden nodes to 1 output node (binary classification)

    def forward(self, x):
        # Specify the forward pass, i.e. activation functions.
        x = torch.relu(self.fc1(x))  # activation function
        x = torch.relu(self.fch(x))
        x = torch.sigmoid(self.fc2(x))

        return x


# Parameters (change and add as needed)
learning_rate = 0.8
num_epochs = 200

model = BinClassifierNN()

# choose a loss function for the classification
criterion = nn.BCELoss() 

# choose an optimizer
optimizer = optim.SGD(model.parameters(), lr=learning_rate)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


for epoch in range(num_epochs):
    
    model.train()
    outputs = model(X_train).squeeze()
    loss = criterion(outputs, y_train)

    optimizer.zero_grad() # clear existing gradients
    loss.backward() # backpropagation of loss function to optimise gradient
    optimizer.step() # update parameters using Adam

    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch + 1}/100], Loss: {loss.item():.4f}')

y_train_pred = ((model(X_train).squeeze() > 0.5).float()).detach().numpy()
y_test_pred = ((model(X_test).squeeze() > 0.5).float()).detach().numpy()

print("Accuracy and ROC AUC")
print("Accuracy on training set:", accuracy_score(y_train, y_train_pred))
print("Accuracy on test set:", accuracy_score(y_test, y_test_pred))
print("ROC AUC on training set:", roc_auc_score(y_train, y_train_pred))
print("ROC AUC on test set:", roc_auc_score(y_test, y_test_pred))

Epoch [10/100], Loss: 0.6564
Epoch [20/100], Loss: 0.6061
Epoch [30/100], Loss: 0.5558
Epoch [40/100], Loss: 0.6129
Epoch [50/100], Loss: 0.5122
Epoch [60/100], Loss: 0.4657
Epoch [70/100], Loss: 0.5044
Epoch [80/100], Loss: 0.4398
Epoch [90/100], Loss: 0.4513
Epoch [100/100], Loss: 0.4208
Epoch [110/100], Loss: 0.3671
Epoch [120/100], Loss: 0.4461
Epoch [130/100], Loss: 0.3459
Epoch [140/100], Loss: 1.2852
Epoch [150/100], Loss: 0.3160
Epoch [160/100], Loss: 0.2680
Epoch [170/100], Loss: 0.3569
Epoch [180/100], Loss: 0.2699
Epoch [190/100], Loss: 0.4829
Epoch [200/100], Loss: 0.2421
Accuracy and ROC AUC
Accuracy on training set: 0.9130882858124356
Accuracy on test set: 0.8056318681318682
ROC AUC on training set: 0.9116052032343863
ROC AUC on test set: 0.8040180237828614


In [110]:
# redo with different fingerprints

X = np.stack(df["TopologicalTorsions_1024"].values) # joins multiple arrays along a new axis - builds a proper array from the line-by-line arrays in the df.
y = df["mutagenicity"].to_numpy()

X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32)

class BinClassifierNN(nn.Module):
    def __init__(self):
        super(BinClassifierNN, self).__init__()
        # define your model width and depth below
        self.fc1 = nn.Linear(1024, 4096)  # 1024 input nodes to 4096 hidden nodes (fc: fully connected)
        self.fch = nn.Linear(4096, 4096)  # 4096 hidden nodes to 4096 hidden nodes
        self.fc2 = nn.Linear(4096, 1)  # 4096 hidden nodes to 1 output node (binary classification)

    def forward(self, x):
        # Specify the forward pass, i.e. activation functions.
        x = torch.relu(self.fc1(x))  # activation function
        x = torch.relu(self.fch(x))
        x = torch.sigmoid(self.fc2(x))

        return x


# Parameters (change and add as needed)
learning_rate = 0.8
num_epochs = 200

model = BinClassifierNN()

# choose a loss function for the classification
criterion = nn.BCELoss() 

# choose an optimizer
optimizer = optim.SGD(model.parameters(), lr=learning_rate)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


for epoch in range(num_epochs):
    
    model.train()
    outputs = model(X_train).squeeze()
    loss = criterion(outputs, y_train)

    optimizer.zero_grad() # clear existing gradients
    loss.backward() # backpropagation of loss function to optimise gradient
    optimizer.step() # update parameters using Adam

    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch + 1}/100], Loss: {loss.item():.4f}')

y_train_pred = ((model(X_train).squeeze() > 0.5).float()).detach().numpy()
y_test_pred = ((model(X_test).squeeze() > 0.5).float()).detach().numpy()

print("Accuracy and ROC AUC")
print("Accuracy on training set:", accuracy_score(y_train, y_train_pred))
print("Accuracy on test set:", accuracy_score(y_test, y_test_pred))
print("ROC AUC on training set:", roc_auc_score(y_train, y_train_pred))
print("ROC AUC on test set:", roc_auc_score(y_test, y_test_pred))

Epoch [10/100], Loss: 0.6461
Epoch [20/100], Loss: 0.5930
Epoch [30/100], Loss: 0.5438
Epoch [40/100], Loss: 0.5907
Epoch [50/100], Loss: 0.5457
Epoch [60/100], Loss: 0.5020
Epoch [70/100], Loss: 0.5254
Epoch [80/100], Loss: 0.5125
Epoch [90/100], Loss: 0.4971
Epoch [100/100], Loss: 0.4831
Epoch [110/100], Loss: 0.4704
Epoch [120/100], Loss: 0.4582
Epoch [130/100], Loss: 0.4443
Epoch [140/100], Loss: 0.4278
Epoch [150/100], Loss: 0.4108
Epoch [160/100], Loss: 0.3987
Epoch [170/100], Loss: 0.4019
Epoch [180/100], Loss: 0.4524
Epoch [190/100], Loss: 0.4019
Epoch [200/100], Loss: 0.2898
Accuracy and ROC AUC
Accuracy on training set: 0.8869804190999656
Accuracy on test set: 0.7946428571428571
ROC AUC on training set: 0.8879595358994422
ROC AUC on test set: 0.7959926411774116


#### 8) Discussion points
1) How did your model compare to other simple ML classifiers (all used the Morgan FPs)? Discuss!
- it performed better than most for the Morgan FPs, the only simple ML classifier which performed similarly was the random forest.
2) Did you observe any difference between different fingerprint types?
- the parameters were optimized for the Morgan FPs, so it is hard to say if it could have performed better on the other types, however, the other fingerprints were not that far off with most of them getting a test accuracy around 80 which is not too bad. the parameters couldve been optimized for the other fingerprints, but that would have taken way too long. One thing to notice is that MACCSkeys had by far less over fitting.
3) Did the fingerprint size impact the model prediction? What message is to be learned from this?
- changing the fingerprint size from 2048 to 1024 for both the Morgan FPs and the Topological FPs did not have a large effect. The accuracy got slightly worse as expected. There was also slightly less overfitting which also makes sense as there is less data to overfit with.
4) What were some model parameters for decent performance depending on the fingerprint type? 
- a learning rate of 0.8 and epochs of 200 seemed to have decent results for all of the fingerprint types, there is a bit of overfitting which could be reduced by lowering the number of epochs, but then the test accuracy would also go down.
5) Was overfitting a problem? What approaches did you apply to limit that issue? What else would be possible
- it was not much of a problem, but if it was, i wouldve lowered the number of epochs.
6) Consider the target "mutagenicity" in the context of molecular structure. What does noise mean here? How could you use such a predictive model in the lab? What other data-driven tools could be interesting in this QSAR context?
- the noise is the parts of the fingerprints which have nothing to do with the mutagenicity of the molecule. the noise can also come from the fact that the mutagenicity is either 1 or 0, and if the molecule is a borderline case, it can be mislabled.
7) Why is exporting a full model usually not recommended?
- it might not work anymore if the version of PyTorch changes. It would be more difficult to make changes later. If you want to change anything, the original save becomes unusable.